<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day02-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 2 — In-class discussion problem (1 of 3)

Work this out **by hand in your group first** — then run the code cell
to check your answer before presenting.

## Which gene changed the most?

Two classmates each look for "the most changed gene" in the mouse liver
RNA-seq table from Section 2 (`df`, one row per gene). Both snippets run
without errors:

```python
# snippet A
df.sort_values("log2_FoldChange").iloc[0]

# snippet B
df.loc[df["log2_FoldChange"].abs().idxmax()]
```

**As a group:**

1. Predict what each snippet returns, before running anything.
2. Do they answer the same biological question?
3. Is either of them what a biologist would mean by "the most changed
   gene"? What about the adjusted p-value, and how highly the gene is
   expressed?

Run the cell below to check your answers.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/JoachimGoedhart/VolcaNoseR/master/Becares-diffgenes_HFHC.csv"
df = pd.read_csv(url)
cols = ["Gene", "log2_FoldChange", "padj", "baseMean"]

a = df.sort_values("log2_FoldChange").iloc[0]
b = df.loc[df["log2_FoldChange"].abs().idxmax()]
c = df.loc[df["padj"].idxmin()]
for name, row in [("A: sort_values().iloc[0]", a), ("B: abs().idxmax()", b), ("most significant (lowest padj)", c)]:
    print(f"{name:32s} {row['Gene']:8s} log2 FC {row['log2_FoldChange']:+.2f} "
          f"({2 ** abs(row['log2_FoldChange']):.1f}-fold)  padj {row['padj']:.1e}  mean expression {row['baseMean']:.0f}")

top = df.reindex(df["log2_FoldChange"].abs().sort_values(ascending=False).index)
print("\nthe eight largest changes in either direction:")
print(top.head(8)[cols].to_string(index=False, formatters={"log2_FoldChange": "{:+.2f}".format, "padj": "{:.1e}".format, "baseMean": "{:.0f}".format}))
print(f"\nmedian mean expression: all genes {df['baseMean'].median():.0f}, "
      f"100 largest changes {top.head(100)['baseMean'].median():.0f}")

A: sort_values().iloc[0]         Syngr1   log2 FC -2.16 (4.5-fold)  padj 1.8e-14  mean expression 99
B: abs().idxmax()                Cyp2c69  log2 FC +2.39 (5.2-fold)  padj 1.8e-14  mean expression 246
most significant (lowest padj)   Osbpl3   log2 FC -1.63 (3.1-fold)  padj 8.1e-18  mean expression 388

the eight largest changes in either direction:
   Gene log2_FoldChange    padj baseMean
Cyp2c69           +2.39 1.8e-14      246
 Syngr1           -2.16 1.8e-14       99
 Elovl3           +2.16 1.4e-10       71
 Mrgpre           -2.14 9.5e-12       52
  Lrtm2           -1.95 1.2e-10      214
   Spp1           -1.93 9.1e-15     2548
  Fabp5           -1.92 1.8e-14      637
  Ces1f           +1.80 2.5e-14      749

median mean expression: all genes 329, 100 largest changes 123


**Discussion point:** both snippets are correct Python, and they answer
different questions.

- **Snippet A** sorts in *ascending* order (the default), so `.iloc[0]`
  is the most *down*-regulated gene: *Syngr1*, 4.5-fold down. It is "the
  most changed gene" only if you meant "the most decreased".
- **Snippet B** takes the largest change in *either* direction:
  *Cyp2c69*, 5.2-fold up.

And "most changed" is still ambiguous. The most *significant* gene is a
third one, *Osbpl3* (adjusted $p = 8 \times 10^{-18}$, 3.1-fold down).
The largest fold changes also come mostly from lowly expressed genes
(median mean expression 123 among the 100 largest changes, against 329
overall), where estimates are noisiest (the Section 3 plot).

This is a mistake of the second kind: the code runs, the output looks
reasonable, and nothing warns you that it answers a different question.
An AI assistant asked for "the most changed gene" will happily write
either snippet. **Decide the biological question first, then write the
code.**